<a href="https://colab.research.google.com/github/MElsdk-lab/Biochar_forest_estimation/blob/main/Notebook_2_Forest_Area_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# NOTEBOOK 2 — Forest Area Analysis & FAO Comparison
# University of Pittsburgh | Biochar Feedstock Methodology
# ============================================================

In [ ]:
# ── CELL 1: Install Libraries ─────────────────────────────────────────────────
!pip install -q pandas matplotlib

#considering that output of notebook 1 are store
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
## ── CELL 2: import Libraries and results & data─────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt
from data_config import FAO_name_fix, fao_fra_2025, build_country_lookup, FAO_LSIB_REGION

print('✅ Libraries imported')
print('✅ Data config loaded')





In [ ]:
## ── CELL 3  : merge results from notebook1 into main file ─────────────────────────────────────────────────
GEE_FOLDER = ('/content/drive/MyDrive/BiocharProject/GEE_exports/')
country_forest_area_10 = pd.read_csv(GEE_FOLDER + 'forest_area_10.csv')
country_forest_area_20= pd.read_csv(GEE_FOLDER + 'forest_area_20.csv')
country_forest_area_30 = pd.read_csv(GEE_FOLDER + 'forest_area_30.csv')
country_forest_area_40 = pd.read_csv(GEE_FOLDER + 'forest_area_40.csv')
country_forest_area_50  = pd.read_csv(GEE_FOLDER + 'forest_area_50.csv')

country_total_forest_area =pd.concat([country_forest_area_10, country_forest_area_20, country_forest_area_30, country_forest_area_40, country_forest_area_50], ignore_index = True)
country_total_forest_area = country_total_forest_area.groupby(['country_na', 'threshold'])['sum'].sum().reset_index()
country_total_forest_area = country_total_forest_area.rename(columns={'country_na': 'country', 'sum': 'total_forest_area'})

state_forest_area_10 = pd.read_csv(GEE_FOLDER + 'states_forest_area_10.csv')
state_forest_area_20 = pd.read_csv(GEE_FOLDER + 'states_forest_area_20.csv')
state_forest_area_30 = pd.read_csv(GEE_FOLDER + 'states_forest_area_30.csv')
state_forest_area_40 = pd.read_csv(GEE_FOLDER + 'states_forest_area_40.csv')
state_forest_area_50 = pd.read_csv(GEE_FOLDER + 'states_forest_area_50.csv')

state_total_forest_area = pd.concat([state_forest_area_10, state_forest_area_20, state_forest_area_30, state_forest_area_40, state_forest_area_50], ignore_index = True)
state_total_forest_area = state_total_forest_area.groupby(['NAME', 'threshold'])['sum'].sum().reset_index()
state_total_forest_area = state_total_forest_area.rename(columns={'NAME': 'state','sum': 'total_forest_area'})

In [ ]:
## ── CELL 4  : build region and subregion info for countries in 'country_total_forest_area' file by the intermidate of FAO_LSIB_REGION   ─────────────────────────────────────────────────
# 1. Generate the lookup dictionary
country_lookup = build_country_lookup(FAO_LSIB_REGION)

# 2. Create the new columns by looking up each country name
# we could actualy transform the country_lookup from a dictionary to a data frame then merge with df bsed onthe country name
country_total_forest_area  ['region'] = country_total_forest_area['country'].map(lambda l: country_lookup.get(l, {}).get('region', 'unknown'))
country_total_forest_area  ['subregion'] = country_total_forest_area['country'].map(lambda l: country_lookup.get(l, {}).get('subregion', 'unknown'))

print(country_total_forest_area.head(3))

In [4]:
# ── CELL 5: Clean FAO FRA data ───────────────────────────
fao_fra_2025['country'] = fao_fra_2025['country'].replace(FAO_name_fix)

# add region + subregion to FAO data
fao_fra_2025['region'] = fao_fra_2025['country'].map(
    lambda l: country_lookup.get(l, {}).get('region', 'unknown'))

fao_fra_2025['subregion'] = fao_fra_2025['country'].map(
    lambda l: country_lookup.get(l, {}).get('subregion', 'unknown'))

print(fao_fra_2025[['country', 'region', 'subregion', '2000']].head())

# convert all years from 1000 ha to Mha
year_columns = ['1990', '2000', '2010', '2015', '2020', '2025']
fao_fra_2025[year_columns] = fao_fra_2025[year_columns] / 1000

print(fao_fra_2025.head())

In [ ]:
# ── CELL 6: Merge GEE results with FAO 2000 ──────────────
FAO_forest_area_country_2000 = fao_fra_2025[['country', '2000']].rename(columns={'2000': 'FAO_2000_Mha'})

gee_fao_comparison = country_total_forest_area.merge(FAO_forest_area_country_2000, on='country', how='left')

print(gee_fao_comparison.head())
print(f'Total countries: {gee_fao_comparison["country"].nunique()}')